In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# MIXTURE OF EXPERTS LAYER (SIMPLIFIED DEEPSEEK V3 STYLE)
# Key insight: N experts but only K active per token
# ============================================================

class SwiGLUExpert(nn.Module):
    """Single expert FFN with SwiGLU activation (used by Llama, DeepSeek)."""
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden_dim, bias=False)  # Gate projection
        self.w2 = nn.Linear(hidden_dim, dim, bias=False)  # Down projection
        self.w3 = nn.Linear(dim, hidden_dim, bias=False)  # Up projection

    def forward(self, x):
        # SwiGLU: combines gating with feedforward
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

class MoELayer(nn.Module):
    """
    Mixture of Experts layer with top-k routing.

    Architecture (DeepSeek V3 simplified):
 - 1 shared expert (always active)
 - N routed experts (top-K selected per token)
 - Router: simple linear layer -> softmax -> top-K
    """
    def __init__(self, dim, hidden_dim, num_experts=8, top_k=2):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k

        # Router: produces a score for each expert
        self.router = nn.Linear(dim, num_experts, bias=False)

        # Shared expert (always active, like DeepSeek V3)
        self.shared_expert = SwiGLUExpert(dim, hidden_dim)

        # Routed experts (only top-K active per token)
        self.experts = nn.ModuleList([
            SwiGLUExpert(dim, hidden_dim) for _ in range(num_experts)
        ])

    def forward(self, x, return_aux_loss=True):
        batch, seq_len, dim = x.shape
        x_flat = x.view(-1, dim)  # [B*S, D]

        # Step 1: Router scores
        router_logits = self.router(x_flat)  # [B*S, N]
        router_probs = F.softmax(router_logits, dim=-1)

        # Step 2: Top-K selection
        top_k_probs, top_k_idx = torch.topk(
            router_probs, self.top_k, dim=-1
        )
        # Normalize selected weights to sum to 1
        top_k_weights = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)

        # Step 3: Shared expert (always runs)
        shared_out = self.shared_expert(x_flat)

        # Step 4: Routed experts (only top-K run)
        routed_out = torch.zeros_like(x_flat)
        for i, expert in enumerate(self.experts):
            # Find tokens routed to this expert
            mask = (top_k_idx == i).any(dim=-1)
            if mask.any():
                expert_out = expert(x_flat[mask])
                # Weight by gate probability
                expert_weight = top_k_weights[mask]
                idx_match = (top_k_idx[mask] == i).float()
                weight = (expert_weight * idx_match).sum(dim=-1, keepdim=True)
                routed_out[mask] += weight * expert_out

        # Combine shared + routed
        output = shared_out + routed_out

        # Step 5: Auxiliary load balancing loss
        aux_loss = None
        if return_aux_loss:
            # f_i = fraction of tokens routed to each expert
            expert_counts = torch.zeros(self.num_experts, device=x.device)
            for i in range(self.num_experts):
                expert_counts[i] = (top_k_idx == i).float().sum()
            f_i = expert_counts / (x_flat.shape[0] * self.top_k)

            # P_i = average router probability per expert
            P_i = router_probs.mean(dim=0)

            # L_aux = alpha * N * sum(f_i * P_i)
            aux_loss = self.num_experts * (f_i * P_i).sum()

        return output.view(batch, seq_len, dim), aux_loss

# ============================================================
# DEMO: Compare dense vs MoE
# ============================================================
dim, hidden = 512, 1024  # Small for demo
num_experts, top_k = 8, 2

dense = SwiGLUExpert(dim, hidden)
moe = MoELayer(dim, hidden, num_experts, top_k)

dense_params = sum(p.numel() for p in dense.parameters())
moe_params = sum(p.numel() for p in moe.parameters())
# Active = shared_expert + top_k * expert_size + router
active_params = dense_params + top_k * dense_params + dim * num_experts

print("=" * 55)
print("DENSE vs MoE COMPARISON")
print("=" * 55)
print(f"Dense FFN params:     {dense_params:,}")
print(f"MoE total params:     {moe_params:,}")
print(f"MoE active per token: ~{active_params:,}")
print(f"Capacity multiplier:  {moe_params / dense_params:.1f}x")
print(f"Compute multiplier:   {active_params / dense_params:.1f}x")

# Test forward pass
x = torch.randn(2, 10, dim)
out, aux = moe(x)
print(f"\nInput:  {x.shape}")
print(f"Output: {out.shape}")
print(f"Aux loss: {aux.item():.4f} (lower = more balanced)")

DENSE vs MoE COMPARISON
Dense FFN params:     1,572,864
MoE total params:     14,159,872
MoE active per token: ~4,722,688
Capacity multiplier:  9.0x
Compute multiplier:   3.0x

Input:  torch.Size([2, 10, 512])
Output: torch.Size([2, 10, 512])
Aux loss: 1.0522 (lower = more balanced)
